In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Funcția IBM Circuit

> **Note:** * Qiskit Functions este o funcționalitate experimentală disponibilă doar utilizatorilor IBM Quantum&reg; Premium Plan, Flex Plan și On-Prem (prin IBM Quantum Platform API) Plan. Se află în stare de previzualizare și poate fi supusă modificărilor.

## Prezentare generală
Funcția IBM&reg; Circuit primește [PUB-uri abstracte](./primitive-input-output) ca intrări și returnează valori de așteptare atenuate ca ieșiri. Această funcție Circuit include un pipeline automatizat și personalizat pentru a le permite cercetătorilor să se concentreze pe descoperirea de algoritmi și aplicații.
## Descriere
După ce trimiți PUB-ul tău, circuitele și observabilele abstracte sunt transpilate automat, executate pe hardware și post-procesate pentru a returna valori de așteptare atenuate. În acest scop, sunt combinate următoarele instrumente:

- [Qiskit Transpiler Service](./qiskit-transpiler-service), inclusiv selecția automată a paselor de transpilare bazate pe AI și euristice pentru a traduce circuitele abstracte în circuite ISA optimizate pentru hardware
- [Suprimarea și atenuarea erorilor necesare pentru calculul la scară utilitară](./error-mitigation-and-suppression-techniques), inclusiv twirling-ul măsurătorilor și al porților, decuplarea dinamică, Twirled Readout Error eXtinction (TREX), Zero-Noise Extrapolation (ZNE) și Probabilistic Error Amplification (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), pentru a executa PUB-uri ISA pe hardware și a returna valori de așteptare atenuate

![Funcția IBM Circuit](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)
## Începe
Autentifică-te folosind [cheia ta API](http://quantum.cloud.ibm.com/) și selectează Qiskit Function după cum urmează. (Acest fragment de cod presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## Exemplu
 Pentru a începe, încearcă acest exemplu de bază:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Verifică [statusul](/guides/functions#check-job-status) workload-ului Qiskit Function sau returnează [rezultatele](/guides/functions#retrieve-results) după cum urmează:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


Rezultatele au același format ca un [rezultat Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## Intrări
Consultă tabelul următor pentru toți parametrii de intrare acceptați de funcția IBM Circuit. Secțiunea [Opțiuni](#options) de mai jos oferă mai multe detalii despre `options` disponibile.
| Nume      | Tip                       | Descriere                                                                                                                                                                                                                         | Obligatoriu | Implicit                                                                  | Exemplu                                  |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | Numele Backend-ului pentru care se face interogarea.                                                                                                                                                                                              | Da      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | Un iterabil de obiecte PUB-like abstracte (primitive unified bloc), cum ar fi tupluri `(circuit, observables)` sau `(circuit, observables, parameter_values)`. Vezi [Prezentare generală a PUB-urilor](/guides/primitive-input-output#overview-of-pubs) pentru mai multe informații. Circuitele pot fi abstracte (non-ISA). | Da      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | Opțiuni de intrare. Vezi secțiunea **Opțiuni** pentru mai multe detalii.                                                                                                                                                                                | Nu       | Vezi secțiunea **Opțiuni** pentru detalii.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | Numele resursei cloud al instanței de utilizat în acel format.                                                                                                                                                                                        | Nu       | Una este aleasă aleatoriu dacă contul tău are acces la mai multe instanțe. | `CRN`                   |
### Opțiuni
#### Structură
Similar cu primitivele Qiskit Runtime, opțiunile pentru funcția IBM Circuit pot fi specificate ca un dicționar imbricat. Opțiunile frecvent utilizate, cum ar fi ``optimization_level`` și ``mitigation_level``, se află la primul nivel. Alte opțiuni mai avansate sunt grupate în diferite categorii, cum ar fi ``resilience``.

#### Valori implicite
Dacă nu specifici o valoare pentru o opțiune, se utilizează valoarea implicită specificată de serviciu.

#### Nivelul de atenuare
Funcția IBM Circuit suportă de asemenea `mitigation_level`. Nivelul de atenuare specifică cât de multă suprimare și atenuare a erorilor să se aplice jobului. Nivelurile mai ridicate generează rezultate mai precise, cu costul unor timpi de procesare mai lungi. Gradul de reducere a erorilor depinde de metoda aplicată. Nivelul de atenuare abstractizează alegerea detaliată a metodelor de atenuare și suprimare a erorilor pentru a permite utilizatorilor să raționeze despre compromisul cost/precizie potrivit aplicației lor. Tabelul următor arată metodele corespunzătoare pentru fiecare nivel.

> **Note:** Deși numele sunt similare, `mitigation_level` aplică tehnici diferite față de cele utilizate de `resilience_level` al Estimator.

Similar cu ``resilience_level`` în Qiskit Runtime Estimator, ``mitigation_level`` specifică un set de bază de opțiuni selectate. Orice opțiuni pe care le specifici manual în plus față de nivelul de atenuare sunt aplicate peste setul de bază de opțiuni definit de nivelul de atenuare. Prin urmare, în principiu, ai putea seta nivelul de atenuare la 1, dar apoi să dezactivezi atenuarea măsurătorilor, deși acest lucru nu este recomandat.

| **Nivel de atenuare** | **Tehnică** |
|:-:|:-:|
| 1 [Implicit] | Decuplare dinamică + twirling măsurători + TREX  |
| 2 | Nivelul 1 + twirling porți + ZNE prin plierea porților |
| 3 | Nivelul 1 + twirling porți + ZNE prin PEA |

Exemplul următor demonstrează setarea nivelului de atenuare:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### Toate opțiunile disponibile
Pe lângă `mitigation_level`, funcția IBM Circuit oferă și un număr selectat de opțiuni avansate care îți permit să reglezi fin compromisul cost/precizie. Tabelul următor arată toate opțiunile disponibile:

| Opțiune | Sub-opțiune | Sub-sub-opțiune | Descriere | Valori posibile | Implicit |
|-|-|-|-|-|-|
| default_precision |  |  | Precizia implicită de utilizat pentru orice PUB sau apel `run()`<br/>care nu specifică una.<br/>Fiecare PUB de intrare poate specifica propria precizie. Dacă metoda `run()` primește o precizie, atunci acea valoare este utilizată pentru toate PUB-urile din apelul `run()` care nu specifică propria precizie.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Timpul maxim de execuție în secunde, care se bazează<br/>pe utilizarea QPU (nu pe timpul de ceas). Utilizarea QPU reprezintă<br/>timpul în care QPU este dedicat procesării jobului tău. Dacă un job depășește această limită de timp, este anulat forțat. | Număr întreg de secunde în intervalul [1, 10800] |  |
| mitigation_level |  |  | Cât de multă suprimare și atenuare a erorilor să se aplice. Consultă secțiunea [Nivelul de atenuare](#mitigation-level) pentru mai multe informații despre metodele utilizate la fiecare nivel. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Cât de multă optimizare să se efectueze pe circuite. [Nivelurile mai ridicate](/guides/set-optimization) generează circuite mai optimizate, cu costul unui timp de transpilare mai lung. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Dacă să se activeze decuplarea dinamică. Consultă [Tehnici de suprimare și atenuare a erorilor](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) pentru explicarea metodei.  | True/False | True |
|  | sequence_type |  | Ce secvență de decuplare dinamică să se utilizeze.<br/>* `XX`: utilizează secvența `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: utilizează secvența `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: utilizează secvența<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Dacă să se aplice twirling-ul porților Clifford cu 2 Qubiți. | True/False | False |
|  | enable_measure |  | Dacă să se activeze twirling-ul măsurătorilor. | True/False | True |
| resilience | measure_mitigation |  | Dacă să se activeze metoda de atenuare a erorilor de măsurare TREX. Consultă [Tehnici de suprimare și atenuare a erorilor](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) pentru explicarea metodei.  | True/False | True |
|  | zne_mitigation |  | Dacă să se activeze metoda de atenuare a erorilor Zero Noise Extrapolation. Consultă [Tehnici de suprimare și atenuare a erorilor](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) pentru explicarea metodei.  | True/False | False |
|  | zne | amplifier | Ce tehnică să se utilizeze pentru amplificarea zgomotului. Una dintre: <br/> - `gate_folding` (implicit) utilizează plierea porților cu 2 Qubiți pentru a amplifica zgomotul. Dacă factorul de zgomot necesită amplificarea doar a unui subset de porți, atunci aceste porți sunt alese aleatoriu.<br/><br/> - `gate_folding_front` utilizează plierea porților cu 2 Qubiți pentru a amplifica zgomotul. Dacă factorul de zgomot necesită amplificarea doar a unui subset de porți, atunci aceste porți sunt selectate de la începutul circuitului DAG ordonat topologic.<br/><br/> - `gate_folding_back` utilizează plierea porților cu 2 Qubiți pentru a amplifica zgomotul. Dacă factorul de zgomot necesită amplificarea doar a unui subset de porți, atunci aceste porți sunt selectate de la sfârșitul circuitului DAG ordonat topologic.<br/><br/> - `pea` utilizează o tehnică numită Probabilistic error amplification (PEA) pentru a amplifica zgomotul. Consultă [Tehnici de suprimare și atenuare a erorilor](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) pentru explicarea metodei.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Factorii de zgomot de utilizat pentru amplificarea zgomotului. | listă de numere float; fiecare float >= 1 | (1, 1.5, 2) pentru PEA, și (1, 3, 5) în alte cazuri. |
|  |  | extrapolator | Factorii de zgomot la care se evaluează modelele de extrapolare. Această opțiune nu afectează în niciun fel execuția sau ajustarea modelului; determină doar punctele în care obiectele `extrapolator` sunt evaluate pentru a fi returnate în câmpurile de date numite `evs_extrapolated` și `stds_extrapolated`. | unul sau mai multe dintre `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Dacă să se activeze metoda de atenuare a erorilor Probabilistic Error Cancellation. Consultă [Tehnici de suprimare și atenuare a erorilor](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) pentru explicarea metodei.  | True/False | False |
|  | pec | max_overhead | Supraîncărcarea maximă admisă la eșantionarea circuitelor, sau `None` pentru nicio limită maximă. | None/ integer >1 | 100 |

În exemplul următor, setarea nivelului de atenuare la 1 dezactivează inițial atenuarea ZNE, dar setarea `zne_mitigation` la `True` suprascrie configurarea relevantă din `mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## Ieșiri
Ieșirea unei funcții Circuit este un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), care conține două câmpuri:

- Unul sau mai multe obiecte [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Acestea pot fi indexate direct din `PrimitiveResult`.

- Metadate la nivel de job.

Fiecare `PubResult` conține un câmp `data` și un câmp `metadata`.

- Câmpul `data` conține cel puțin o serie de valori de așteptare (`PubResult.data.evs`) și o serie de erori standard (`PubResult.data.stds`). Poate conține de asemenea mai multe date, în funcție de opțiunile utilizate.

- Câmpul `metadata` conține metadate la nivel de PUB (`PubResult.metadata`).

Fragmentul de cod următor descrie formatul `PrimitiveResult` (și `PubResult` asociat).